# LFM3 — fine-tuning Kali sur T4

Base : **OpenIntelligenceNet/LFM3-1.7B-Uncensored**. Corpus : **5 948 exemples**,
313 exemples de validation et un ensemble de test séparé. QLoRA 4 bits, rang 16.

Exécuter les cellules dans l'ordre. Autoriser le montage de Google Drive pour
conserver les sauvegardes. La cellule d'entraînement reprend automatiquement
le dernier checkpoint de `MyDrive/LFM3_Kali_Finetune/run_20261105_v1`.

La première exécution télécharge environ 3,46 Go de poids. Le calcul est un
fine-tuning des adaptateurs ; seuls les tokens assistant sont supervisés.
Les commandes des datasets ne sont jamais exécutées. La dernière cellule
compare les réponses du modèle de base et du modèle spécialisé.

Une fermeture ou déconnexion peut interrompre la session Colab ; les checkpoints
sur Drive permettent de reprendre. Aucun résultat final n'est prérempli ici.


In [ ]:
import platform
import subprocess
import sys

print("Python :", platform.python_version())
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

import torch

print("PyTorch :", torch.__version__, "CUDA :", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Sélectionner GPU T4 dans Exécution > Modifier le type d'exécution.")
print("GPU détecté :", torch.cuda.get_device_name(0))
print("Installation des bibliothèques de fine-tuning…", flush=True)

# Dans le notebook, cette ligne est convertie en %pip.
%pip install -q "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" "datasets==5.1.0"
print("Installation terminée. Passer à la préparation des données.")


In [ ]:
# Paramètres de la première version. Le contexte d'entraînement est distinct du contexte d'inférence.
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

WORK_DIR = Path("/content/lfm3_cyber_ft")
SAVE_DIR = Path("/content/drive/MyDrive/LFM3_Kali_Finetune/run_20261105_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["LFM3_WORK_DIR"] = str(WORK_DIR)
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

MODEL_ID = "OpenIntelligenceNet/LFM3-1.7B-Uncensored"
MODEL_REVISION = "8e7cf5cccd95afa9d34a3a66ebfb8fef1b55c204"
MAX_LENGTH = 1536
LORA_RANK = 16
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION = 8
EPOCHS = 1
USE_4BIT = True
print("Dossier des sauvegardes :", SAVE_DIR)
print("Modèle :", MODEL_ID)


In [ ]:
"""Assemble un corpus traçable. Les commandes sont des données, jamais exécutées."""

import collections
import hashlib
import json
import os
import random
import re
import shlex
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

SEED = 42
MODEL_ID = "OpenIntelligenceNet/LFM3-1.7B-Uncensored"
MODEL_REVISION = "8e7cf5cccd95afa9d34a3a66ebfb8fef1b55c204"
ROOT = Path(os.environ.get("LFM3_WORK_DIR", "/content/lfm3_cyber_ft"))
RAW = ROOT / "raw"
DATA = ROOT / "data"
SYSTEM = (
    "You are an internal cybersecurity assistant. Select tools and arguments from the request "
    "and available evidence. Use the supplied tool schemas. Never invent tool results or claim "
    "to have executed a command without a tool response. Reply in the user's language."
)
SHELL_TOOL = [{"type": "function", "function": {
    "name": "run_shell", "description": "Execute a Kali/Linux command in the working environment.",
    "parameters": {"type": "object", "properties": {
        "command": {"type": "string", "description": "The exact shell command to execute."}
    }, "required": ["command"], "additionalProperties": False}
}}]

SOURCES = {
    "kali_train.jsonl": {
        "url": "https://raw.githubusercontent.com/RISys-Lab/KaliBench/main/KaliBench_data/kalibench_verified_train_3504.jsonl",
        "sha256": "93dc4ddff6d9df73bed4b9e92716470e7a73df513f5ca2f1b201c85f2329f909", "license": "CC-BY-4.0",
    },
    "kali_test.jsonl": {
        "url": "https://raw.githubusercontent.com/RISys-Lab/KaliBench/main/KaliBench_data/kalibench_verified_test_5000.jsonl",
        "sha256": "783193aaa98762ca384f028b7a7c309994c879ebf0fc1b781dcb7169fff89afc", "license": "CC-BY-4.0",
    },
    "kali_usage.jsonl": {
        "url": "https://raw.githubusercontent.com/RISys-Lab/KaliBench/main/KaliBench_data/Kali_Tool_Subtools_UsageCode.jsonl",
        "sha256": "4515fc581d624652297344f036cd7f429e8002356956b7b528220097a60254cd", "license": "See upstream Kali tool documentation",
    },
    "agent_train.jsonl": {
        "url": "https://huggingface.co/datasets/hotdogs/cyber-sft-agent-qwen38/resolve/3f3d5a112b8739dc76e77ff771f3df526ef246a4/train.jsonl",
        "sha256": "52f3b1ebd0b8e17aa5c74956b8eb9937b93b436b1daa27e58ab9aead94a649d7", "license": "Apache-2.0",
    },
    "agent_valid.jsonl": {
        "url": "https://huggingface.co/datasets/hotdogs/cyber-sft-agent-qwen38/resolve/3f3d5a112b8739dc76e77ff771f3df526ef246a4/valid.jsonl",
        "sha256": "e7dfdba23fd89552fe73d6cec8e4b7ae29fe9d0f26889a55dd732f6e34e01626", "license": "Apache-2.0",
    },
    "agent_test.jsonl": {
        "url": "https://huggingface.co/datasets/hotdogs/cyber-sft-agent-qwen38/resolve/3f3d5a112b8739dc76e77ff771f3df526ef246a4/test.jsonl",
        "sha256": "76bf523273834a76b12e3fe8a7e90abb97a1f7ca5a86e24dd20758ee87fecc57", "license": "Apache-2.0",
    },
    "hackmentor.json": {
        "url": "https://raw.githubusercontent.com/tmylla/HackMentor/main/data/seed_generation-14k.json",
        "sha256": "8f819f6b7336db5cf4404e2b3f54af36daf961c28bdf253da6c4ef09acc5520b", "license": "Apache-2.0 (upstream repository)",
    },
}


def fetch(name):
    meta = SOURCES[name]
    path = RAW / name
    if not path.exists():
        with urllib.request.urlopen(meta["url"], timeout=120) as response:
            payload = response.read()
        path.write_bytes(payload)
    payload = path.read_bytes()
    actual = hashlib.sha256(payload).hexdigest()
    if actual != meta["sha256"]:
        raise ValueError(f"Source modifiée : {name}, attendu {meta['sha256']}, obtenu {actual}")
    print(f"Source vérifiée : {name} ({len(payload)/1e6:.1f} Mo)", flush=True)
    return path


def rows(name):
    text = (RAW / name).read_text(encoding="utf-8")
    return json.loads(text) if name.endswith(".json") else [json.loads(x) for x in text.splitlines() if x.strip()]


def shell_example(question, command, source, key):
    return {
        "source": source, "source_id": str(key), "tools": SHELL_TOOL,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question},
            {"role": "assistant", "content": "", "tool_calls": [
                {"type": "function", "function": {"name": "run_shell", "arguments": {"command": command}}}
            ]},
        ],
    }


def normalize_query(text):
    return re.sub(r"\s+", " ", str(text)).strip().casefold()


def query_key(row):
    return normalize_query(next(m["content"] for m in row["messages"] if m["role"] == "user"))


def make_kali(name, rejected):
    result = []
    for row in rows(name):
        command = row.get("ground_truth_command", "").strip()
        query = row.get("query", "").strip()
        verdict = (row.get("model_verification", {}).get("model_output", {}) or {}).get("verdict")
        if not command or not query or verdict != "ACCURATE":
            rejected["kali_incomplete_or_unverified"] += 1
            continue
        try:
            shlex.split(command)
        except ValueError:
            rejected["kali_unbalanced_shell_quotes"] += 1
            continue
        result.append(shell_example(query, command, "KaliBench", row["custom_id"]))
    return result


CORE_TOOLS = {"nmap", "masscan", "curl", "whatweb", "nikto", "ffuf", "gobuster", "dirb", "wpscan", "searchsploit", "john", "hashcat", "hydra", "sqlmap", "enum4linux"}


def first_tool_calls(name, rejected):
    """Apprend le premier appel uniquement : aucune sortie synthétique contradictoire."""
    result = []
    for row in rows(name):
        if row.get("kind") != "tool":
            continue
        messages = row.get("messages", [])
        first = next((i for i, m in enumerate(messages) if m.get("role") == "assistant"), None)
        if first is None:
            continue
        message = messages[first]
        tool_calls = message.get("tool_calls", [])
        if len(tool_calls) != 1:
            continue
        call = tool_calls[0].get("function", tool_calls[0])
        name_call, args = call.get("name"), call.get("arguments")
        definitions = [x.get("function", x) for x in row.get("tools", [])]
        definition = next((x for x in definitions if x.get("name") == name_call), None)
        if name_call not in CORE_TOOLS or not isinstance(args, dict) or definition is None:
            continue
        schema = definition.get("parameters", {})
        props = schema.get("properties", {})
        if set(args) - set(props) or any(x not in args for x in schema.get("required", [])):
            rejected["agent_invalid_arguments"] += 1
            continue
        if any(props[k].get("type") == "string" and not isinstance(v, str) for k, v in args.items()):
            rejected["agent_wrong_argument_type"] += 1
            continue
        user_messages = [{"role": "user", "content": str(x.get("content", ""))} for x in messages[:first] if x.get("role") == "user"]
        if not user_messages:
            continue
        result.append({
            "source": "Cyber-SFT-Agent:first_call_only", "source_id": row["id"],
            "tools": row.get("tools", []),
            "messages": [{"role": "system", "content": SYSTEM}] + user_messages + [
                {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": name_call, "arguments": args}}]}
            ],
        })
    return result


def hackmentor_qa(rejected):
    result = []
    for i, row in enumerate(rows("hackmentor.json")):
        instruction = str(row.get("instruction", "")).strip()
        question = str(row.get("input", "")).strip() or instruction
        answer = str(row.get("output", "")).strip()
        # Questions explicatives seulement ; évite les faux comptes rendus d'audits exécutés.
        if not re.match(r"(?:what\b|explain\b|describe\b|compare\b|define\b|how (?:does|do|can|is|are)\b)", question, re.I):
            continue
        if not 80 <= len(answer) <= 2200 or re.search(r"after conducting|i(?:'ve| have) (?:found|identified|conducted|scanned)|the scan (?:revealed|identified)", answer, re.I):
            rejected["hackmentor_unsupported_audit_claim"] += 1
            continue
        result.append({"source": "HackMentor:explanations", "source_id": str(i), "tools": [], "messages": [
            {"role": "system", "content": SYSTEM}, {"role": "user", "content": question}, {"role": "assistant", "content": answer}
        ]})
    random.Random(SEED).shuffle(result)
    return result[:1000]


def french_examples():
    result = []
    targets = ["192.0.2.10", "192.0.2.20", "198.51.100.10", "198.51.100.20", "203.0.113.10", "203.0.113.20"]
    ports = ["22,80,443", "80,443", "22,8080", "21,22,25", "53,80,443"]
    for ip in targets:
        for port_list in ports:
            pairs = [
                (f"Avec Nmap, teste en connexion TCP les ports {port_list} de {ip} et affiche les ports ouverts.", f"nmap -sT --open -p {port_list} {ip}"),
                (f"Identifie avec Nmap les versions des services sur les ports {port_list} de {ip}.", f"nmap -sV -p {port_list} {ip}"),
                (f"Affiche uniquement les en-têtes HTTP du serveur http://{ip}/ avec curl.", f"curl -I http://{ip}/"),
                (f"Identifie les technologies du site http://{ip}/ avec WhatWeb.", f"whatweb http://{ip}/"),
                (f"Capture 20 paquets pour l'hôte {ip} sur eth0 avec tcpdump, sans résolution de noms.", f"tcpdump -i eth0 -nn -c 20 host {ip}"),
                (f"Avec tshark, lis capture.pcap et filtre l'adresse IP {ip}.", f"tshark -r capture.pcap -Y 'ip.addr == {ip}'"),
                (f"Liste les suites cryptographiques du service TLS de {ip} sur le port 443 avec Nmap.", f"nmap -sV -p 443 --script ssl-enum-ciphers {ip}"),
            ]
            for question, command in pairs:
                result.append(shell_example(question, command, "FR:curated_tool_templates", len(result)))
    for domain in ["service.test", "audit.test", "interne.test", "laboratoire.test"]:
        for kind in ["A", "AAAA", "MX", "TXT", "NS"]:
            result.append(shell_example(f"Affiche les enregistrements DNS {kind} de {domain} avec dig, en réponse courte.", f"dig +short {domain} {kind}", "FR:curated_tool_templates", len(result)))
    return result


def write_jsonl(path, items):
    with path.open("w", encoding="utf-8") as stream:
        for row in items:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def prepare():
    RAW.mkdir(parents=True, exist_ok=True)
    DATA.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=5) as pool:
        list(pool.map(fetch, SOURCES))
    rejected = collections.Counter()
    kali_train = make_kali("kali_train.jsonl", rejected)
    kali_test = make_kali("kali_test.jsonl", rejected)
    candidates = kali_train + first_tool_calls("agent_train.jsonl", rejected) + hackmentor_qa(rejected) + french_examples()
    protected = kali_test + first_tool_calls("agent_valid.jsonl", rejected) + first_tool_calls("agent_test.jsonl", rejected)
    protected_keys = {query_key(row) for row in protected}
    dedup, seen = [], set()
    for row in candidates:
        key = query_key(row)
        if key in protected_keys or key in seen:
            rejected["duplicate_or_eval_query"] += 1
            continue
        seen.add(key)
        dedup.append(row)
    train, validation = [], []
    by_source = collections.defaultdict(list)
    for row in dedup:
        by_source[row["source"]].append(row)
    rng = random.Random(SEED)
    for source, group in sorted(by_source.items()):
        rng.shuffle(group)
        count = max(1, round(len(group) * 0.05))
        validation.extend(group[:count])
        train.extend(group[count:])
    rng.shuffle(train)
    write_jsonl(DATA / "train.jsonl", train)
    write_jsonl(DATA / "validation.jsonl", validation)
    write_jsonl(DATA / "heldout.jsonl", protected)
    # Les pages d'aide sont conservées pour un RAG ultérieur, pas utilisées comme réponses d'exécution.
    write_jsonl(DATA / "tool_documentation.jsonl", rows("kali_usage.jsonl"))
    assert not ({query_key(x) for x in train} & {query_key(x) for x in validation + protected})
    manifest = {
        "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
        "seed": SEED, "train": len(train), "validation": len(validation), "heldout": len(protected),
        "training_sources": dict(collections.Counter(x["source"] for x in train)),
        "rejected": dict(rejected), "sources": SOURCES,
        "notes": ["No commands are executed by data preparation or training.", "Original evaluation splits never enter training.", "Synthetic agent outputs and reasoning traces are excluded; only schema-validated first calls are retained.", "No refusal or safety-alignment corpus is added; no zero-refusal guarantee is made.", "French examples use reserved documentation addresses and are generated from deterministic templates."],
    }
    (DATA / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in manifest.items() if k != "sources"}, ensure_ascii=False, indent=2), flush=True)
    return manifest


if __name__ == "__main__":
    prepare()


In [ ]:
"""QLoRA LFM3 : réponses assistant seules, contrôle numérique et reprise de checkpoints."""
import collections
import gc
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    Trainer, TrainerCallback, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

# Une réexécution libère les références à l'ancienne tentative avant de recharger.
for old_name in ("trainer", "model", "train_result"):
    globals().pop(old_name, None)
gc.collect()
torch.cuda.empty_cache()

WORK_DIR = Path(globals().get("WORK_DIR", "/content/lfm3_cyber_ft"))
SAVE_DIR = Path(globals().get("SAVE_DIR", "/content/drive/MyDrive/LFM3_Kali_Finetune/run_20261105_v1"))
MODEL_ID = globals().get("MODEL_ID", "OpenIntelligenceNet/LFM3-1.7B-Uncensored")
MODEL_REVISION = globals().get("MODEL_REVISION", "8e7cf5cccd95afa9d34a3a66ebfb8fef1b55c204")
MAX_LENGTH = int(globals().get("MAX_LENGTH", 1536))
LORA_RANK = int(globals().get("LORA_RANK", 16))
LEARNING_RATE = float(globals().get("LEARNING_RATE", 1e-4))
GRADIENT_ACCUMULATION = int(globals().get("GRADIENT_ACCUMULATION", 8))
EPOCHS = int(globals().get("EPOCHS", 1))
USE_4BIT = bool(globals().get("USE_4BIT", True))
DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = SAVE_DIR / "checkpoints"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("L'entraînement nécessite le GPU T4.")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Monter Google Drive pour sauvegarder et reprendre l'entraînement.")

print("Chargement du tokenizer…", flush=True)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True, token=False, padding_side="right"
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def encode_examples(records):
    accepted, skipped = [], collections.Counter()
    for row in records:
        encoded = tokenizer.apply_chat_template(
            row["messages"], tools=row.get("tools") or None,
            tokenize=True, return_dict=True, return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        mask = encoded.get("assistant_masks", encoded.get("assistant_tokens_mask"))
        if not mask or not any(mask) or len(mask) != len(ids):
            raise RuntimeError("Le template ne fournit pas un masque assistant valide.")
        if len(ids) > MAX_LENGTH:
            skipped["too_long_no_truncation"] += 1
            continue
        labels = [token if yes else -100 for token, yes in zip(ids, mask)]
        accepted.append({"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels})
    return accepted, dict(skipped)


print("Tokenisation au format natif LFM3 ; seuls les tokens assistant sont supervisés…", flush=True)
train_rows = read_jsonl(DATA_DIR / "train.jsonl")
validation_rows = read_jsonl(DATA_DIR / "validation.jsonl")
train_encoded, train_skipped = encode_examples(train_rows)
valid_encoded, valid_skipped = encode_examples(validation_rows)
if not train_encoded or not valid_encoded:
    raise RuntimeError("Corpus tokenisé vide.")


class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


def collate(examples):
    length = math.ceil(max(len(row["input_ids"]) for row in examples) / 8) * 8
    return {
        "input_ids": torch.tensor([x["input_ids"] + [tokenizer.pad_token_id] * (length - len(x["input_ids"])) for x in examples]),
        "attention_mask": torch.tensor([x["attention_mask"] + [0] * (length - len(x["input_ids"])) for x in examples]),
        "labels": torch.tensor([x["labels"] + [-100] * (length - len(x["input_ids"])) for x in examples]),
    }


lengths = sorted(len(row["input_ids"]) for row in train_encoded)
stats = {
    "train_examples": len(train_encoded), "validation_examples": len(valid_encoded),
    "input_tokens": sum(lengths), "supervised_tokens": sum(sum(v != -100 for v in row["labels"]) for row in train_encoded),
    "median_length": lengths[len(lengths)//2], "max_length": max(lengths),
    "skipped_train": train_skipped, "skipped_validation": valid_skipped,
}
(SAVE_DIR / "tokenization.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
for name in ("manifest.json", "train.jsonl", "validation.jsonl", "heldout.jsonl", "tool_documentation.jsonl"):
    shutil.copy2(DATA_DIR / name, SAVE_DIR / name)
print(json.dumps(stats, indent=2), flush=True)

print("Chargement des poids Uncensored en 4 bits…", flush=True)
load_options = {
    "revision": MODEL_REVISION, "trust_remote_code": True, "token": False,
    "dtype": torch.float16, "device_map": {"": 0}, "attn_implementation": "sdpa",
}
if USE_4BIT:
    load_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_options)
model.config.use_cache = False
if USE_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})

available_names = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
targets = [name for name in ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3"] if name in available_names]
if len(targets) < 5:
    raise RuntimeError(f"Modules LoRA inattendus : {targets}")
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=2*LORA_RANK, lora_dropout=0.05,
    target_modules=targets, task_type="CAUSAL_LM", bias="none",
))
model.print_trainable_parameters()
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

# Un vrai forward/backward, sans mise à jour des poids, avant le long traitement.
probe_record = next(row for row in train_encoded if 128 <= len(row["input_ids"]) <= 512)
probe_batch = {k: v.to("cuda") for k, v in collate([probe_record]).items()}
model.train()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.autocast("cuda", dtype=torch.float16):
    probe_loss = model(**probe_batch).loss
if not torch.isfinite(probe_loss):
    raise RuntimeError("Perte non finie avant entraînement.")
probe_loss.backward()
gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
if not gradients or not all(torch.isfinite(g).all().item() for g in gradients):
    raise RuntimeError("Gradients non finis avant entraînement.")
if not any(torch.count_nonzero(g).item() for g in gradients):
    raise RuntimeError("Aucun gradient LoRA non nul.")
torch.cuda.synchronize()
probe = {"loss": float(probe_loss.detach()), "seconds": round(time.perf_counter()-started, 3),
         "peak_gpu_gib": round(torch.cuda.max_memory_reserved()/2**30, 3),
         "tokens": len(probe_record["input_ids"]), "nonzero_gradients_verified": True}
(SAVE_DIR / "gpu_probe.json").write_text(json.dumps(probe, indent=2), encoding="utf-8")
print("FORWARD/BACKWARD VALIDÉ :", json.dumps(probe), flush=True)
model.zero_grad(set_to_none=True)
del gradients, probe_batch, probe_loss
torch.cuda.empty_cache()


class Progress(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        # Premier point de reprise très tôt, puis sauvegardes périodiques.
        if state.global_step == 5:
            control.should_save = True
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        record = dict(logs or {})
        record.update(step=state.global_step, total_steps=state.max_steps,
                      gpu_gib=round(torch.cuda.memory_reserved()/2**30, 2))
        (SAVE_DIR / "progress.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        print("PROGRESSION", json.dumps(record), flush=True)
        for key in ("loss", "grad_norm"):
            if key in record and not math.isfinite(float(record[key])):
                raise RuntimeError(f"Arrêt : {key} non fini")


arguments = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    per_device_eval_batch_size=1, learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine", warmup_steps=20,
    fp16=True, bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch", max_grad_norm=1.0,
    logging_steps=5, logging_first_step=True, logging_nan_inf_filter=False,
    save_strategy="steps", save_steps=50, save_total_limit=3,
    eval_strategy="steps", eval_steps=100,
    report_to="none", dataloader_num_workers=0, seed=42, data_seed=42,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model, args=arguments,
    train_dataset=EncodedDataset(train_encoded), eval_dataset=EncodedDataset(valid_encoded[:64]),
    data_collator=collate, processing_class=tokenizer, callbacks=[Progress()],
)
if trainer.accelerator.scaler is not None:
    # Le T4 calcule en FP16. L'échelle par défaut 65536 déborde au premier pas
    # sur cette architecture ; le forward/backward non amplifié est déjà validé.
    trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
    print("AMP loss scale initial :", trainer.accelerator.scaler.get_scale(), flush=True)
checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print("Reprise :", checkpoint or "nouvel entraînement", flush=True)
print("ENTRAÎNEMENT DÉMARRÉ —", len(train_encoded), "exemples ; sauvegarde à l'étape 5 puis toutes les 50 étapes.", flush=True)
train_result = trainer.train(resume_from_checkpoint=checkpoint)
trainer.save_model(str(SAVE_DIR / "adapter"))
tokenizer.save_pretrained(SAVE_DIR / "adapter")
trainer.save_state()
evaluation = trainer.evaluate()
results = {"training": train_result.metrics, "evaluation": evaluation,
           "peak_gpu_gib": torch.cuda.max_memory_reserved()/2**30,
           "base_model": MODEL_ID, "base_revision": MODEL_REVISION,
           "adapter_path": str(SAVE_DIR / "adapter"), "tokenization": stats}
(SAVE_DIR / "result.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("ENTRAÎNEMENT TERMINÉ :", json.dumps(results, indent=2), flush=True)


In [ ]:
"""Comparaison qualitative sur des requêtes de base ; aucune commande n'est exécutée."""
import json
import time

PROBES = [
    "Affiche la version de Nmap.",
    "Avec Nmap, fais un scan TCP connect des ports 22,80,443 de 192.0.2.30 et affiche seulement les ports ouverts.",
    "Affiche les en-têtes HTTP de http://service.test/ avec curl.",
    "Avec tshark, lis capture.pcap et affiche les paquets DNS.",
    "Explique en français la différence entre un port fermé et un port filtré dans Nmap.",
]
RESULTS = []
model.eval()


@torch.no_grad()
def answer(question):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        tools=SHELL_TOOL, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        ids = model.generate(**inputs, max_new_tokens=160, do_sample=False, use_cache=True,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


for question in PROBES:
    with model.disable_adapter():
        original = answer(question)
    specialized = answer(question)
    RESULTS.append({"question": question, "base": original, "fine_tuned": specialized})
    print("\nDEMANDE :", question)
    print("BASE :", original)
    print("FINE-TUNÉ :", specialized)

(SAVE_DIR / "qualitative_tests.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nComparaison enregistrée :", SAVE_DIR / "qualitative_tests.json")
